# GPU vs FPGA — Rate-Distortion Comparison

Compare inference quality on **GPU** (W&B) and on **FPGA** (ZCU102) across all four architectures
(**ResSHyp**, **SHyp**, **ResFP**, **FP**) — ReLU activation, no output_padding baseline.

**Data sources:**
- **GPU metrics** — loaded from `SAR_DDC_FPGA_all_runs_WandB.csv` (no API call required).
  Regenerate with `python notebooks/fetch_wandb_runs.py` (~1 min).
- **FPGA metrics** — read from `compiled_models/<name>/results/metrics.json` after `batch_deploy.py`.

**Structure:**
1. Setup
2. Load GPU runs → tidy DataFrame
3. Load FPGA results → tidy DataFrame
4. Merge & coverage check
5. Aggregate statistics across seeds
6. RD-curve plots (GPU vs FPGA)
7. FPGA degradation Δ plot
8. BPP comparison (likelihood vs rANS bitstream)
9. Hamburg tile visualization + close-up
10. All-metrics grid

## 1 · Setup & filters

In [ ]:
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(ROOT_DIR))
sys.path.insert(0, str(ROOT_DIR / "notebooks"))

# ── Data sources ──────────────────────────────────────────────────────────────
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"
COMPILED_MODELS_DIR = ROOT_DIR / "results" / "fpga" / "compiled_models"

# ── Seeds ─────────────────────────────────────────────────────────────────────
# Seeds 0-5 are the main training seeds; seed=42 is excluded (manual tests).
ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5]

# ── Shared conventions, palette, loaders ──────────────────────────────────────
from _plotkit import (
    PALETTE,
    ARCH_LABEL,
    export_manuscript,
    load_quality_runs,
    load_fpga_quality,
    compare_at_lambda,
)

# Per-architecture colors — used in RD-curve plots (color encodes architecture)
ARCH_COLORS = {
    arch: PALETTE["architectures"].get(arch, "#999999")
    for arch in ["ResSHyp", "SHyp", "ResFP", "FP"]
}

# Per-backend colors — used in delta, BPP, and tile visualization
BACKEND_COLORS = {
    "gpu": PALETTE["compare_gpu_fpga"]["gpu"],  # "#0072B2" blue
    "fpga": PALETTE["compare_gpu_fpga"]["fpga"],  # "#009E73" green
}

# Per-metric colors — used in delta bar chart
METRIC_COLORS = {
    "psnr": PALETTE["metrics"]["psnr"],
    "ssim": PALETTE["metrics"]["ssim"],
    "epd": PALETTE["metrics"]["epd"],
    "error_bars": PALETTE["metrics"]["error_bars"],
}

# Same marker for both backends — linestyle encodes backend (solid=GPU, dashed=FPGA)
MARKERS = {"gpu": "o", "fpga": "o"}
LINESTYLES = {"gpu": "-", "fpga": "--"}

# Arch display labels — shared with _plotkit
_ARCH_LBL = ARCH_LABEL

# ── Output ────────────────────────────────────────────────────────────────────
PLOTS_DIR = ROOT_DIR / "results" / "plots"
SAVE_FIGURES = True  # global toggle — set False to skip all saves
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Per-architecture production learning rate (lr-sweep result) ───────────────
# Runs are selected by lr VALUE (CSV `model.net_optimizer.lr` → `lr`), never by tag.
# SH/ResSH were retrained: SH 5e-5→5e-4, ResSH 5e-5→1e-4; FP/ResFP stay at 5e-4.
PROD_LR = {"FP": 5e-4, "ResFP": 5e-4, "SHyp": 5e-4, "ResSHyp": 1e-4}

# ── Finalized manuscript figures (PDF) land here; \includegraphics has no
# extension so a PDF overrides the old same-named PNG with no .tex edit. ──
MANUSCRIPT_DIR = ROOT_DIR / "LaTeX" / "SAR_DDC_FPGA_TGRS_2026" / "figures" / "images"

# ── ANSI shortcuts ────────────────────────────────────────────────────────────
r, g, b, y, e = "\033[31m", "\033[32m", "\033[34m", "\033[33m", "\033[0m"

## 2 · Load GPU runs → tidy DataFrame

Loads from the pre-built W&B CSV via `_plotkit.load_quality_runs`.
No API call — regenerate with `python notebooks/fetch_wandb_runs.py` (~1 min).

Filters applied (FPGA-deployable baseline):
- **Dataset**: TSXSSCDataModule | **Seeds**: 0–5 | **Tags**: drops `debug`, `crashed`
- **LR**: production lr by VALUE (`PROD_LR`), never by tag
- **Activation**: ReLU only (`relu_only=True`) | **Output padding**: `no_output_padding=True`

**Tidy / long format** — one row per `(lambda, seed, backend=gpu)`. Makes aggregation and
plotting one-liners instead of nested loops.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_LOAD = ["ResSHyp", "SHyp", "ResFP", "FP"]  # restrict to a subset if needed
# ─────────────────────────────────────────────────────────────────────────────
gpu_df = load_quality_runs(
    WANDB_CSV,
    lr="prod",
    archs=ARCHS_TO_LOAD,
    seeds=ACCEPTED_SEEDS,
    verbose=True,
)
print(f"  {gpu_df['lambda'].nunique()} λ values  ×  {gpu_df['seed'].nunique()} seeds")
gpu_df.head(6)

## 3 · Load FPGA results → tidy DataFrame

Scans `compiled_models/` via `_plotkit.load_fpga_quality`. Each deployed model has a
`manifest.json` (seed, lambda, wandb_run_id) and `results/metrics.json` (INT8 quality).
Only seeds in `ACCEPTED_SEEDS` are included; models without metrics.json are warned and skipped.

**FPGA metrics**: `bpp_bitstream` (real rANS), `psnr`/`ssim`/`epd` vs MERLIN and ADAM.
`bpp_likelihood` and SSIM are `NaN` (not computed for INT8 inference).

In [ ]:
fpga_df = load_fpga_quality(
    gpu_df=gpu_df,
    compiled_dir=COMPILED_MODELS_DIR,
    archs=ARCHS_TO_LOAD,
    seeds=ACCEPTED_SEEDS,
    verbose=True,
)
print(f"  {fpga_df['lambda'].nunique()} λ values  ×  {fpga_df['seed'].nunique()} seeds")
fpga_df.head(6)

## 4 · Merge & coverage check

Concatenate GPU and FPGA rows into one DataFrame and verify we have both backends for
every `(lambda, seed)` pair. Missing combinations are shown explicitly.

**`pivot_table` primer** — reshapes long → wide: `index` defines rows, `columns` defines
the new column headers (here `backend`), `values` is the cell content. Think of it as a
spreadsheet cross-tab. Missing combinations appear as `NaN`.


In [ ]:
# Merge both backends into a single tidy DataFrame
df = pd.concat([gpu_df, fpga_df], ignore_index=True)
df["lambda"] = df["lambda"].astype(float)
df["seed"] = df["seed"].astype(int)
df = df.sort_values(["lambda", "seed", "backend"]).reset_index(drop=True)

print(f"Full DataFrame: {len(df)} rows ({df['backend'].value_counts().to_dict()})")
print(f"λ values: {sorted(df['lambda'].unique())}")
print(f"Seeds:    {sorted(df['seed'].unique())}")

# ── Coverage table ────────────────────────────────────────────────────────────
# pivot_table: rows=lambda, columns=backend, cells=count of runs present (should all be 6)
coverage = df.pivot_table(
    index=["lambda", "arch"], columns="backend", values="psnr_merlin", aggfunc="count"
).rename_axis(None, axis=1)

missing_gpu = coverage[coverage["gpu"] < len(ACCEPTED_SEEDS)] if "gpu" in coverage else []
missing_fpga = coverage[coverage["fpga"] < len(ACCEPTED_SEEDS)] if "fpga" in coverage else []

print(f"\nCoverage (expected {len(ACCEPTED_SEEDS)} runs per λ per backend):")
print(coverage.to_string())
if len(missing_gpu):
    print(f"\n{y}WARNING: incomplete GPU coverage:{e}")
    print(missing_gpu)
if len(missing_fpga):
    print(f"\n{y}WARNING: incomplete FPGA coverage:{e}")
    print(missing_fpga)
else:
    print(f"\n{g}✓ Full coverage: all (λ, seed) pairs present on both backends.{e}")


## 5 · Aggregate statistics across seeds

`groupby(["lambda","backend"]).agg(...)` produces a DataFrame indexed by `(lambda, backend)` with a
two-level column `(metric, stat)`. This is the single source of truth for all plots below.

`query("backend == 'gpu'")` — pandas SQL-style row filter. Returns a view (not a copy), so it's
fast and readable. Equivalent to `df[df["backend"] == "gpu"]` but cleaner for compound conditions.


In [ ]:
METRIC_COLS = [
    "bpp_bitstream",
    "psnr_merlin",
    "mse_merlin",
    "ssim_merlin",
    "ms_ssim_merlin",
    "psnr_adam_noc",
    "mse_adam_noc",
    "ssim_adam_noc",
    "ms_ssim_adam_noc",
    # SAR quality metrics
    "enl_recon",
    "ratio_mean",
    "ratio_enl",
    "epd_merlin",
    "epd_adam_noc",
]

# groupby + agg: for each (lambda, backend) group compute mean/std/min/max of every metric.
# Result is a DataFrame with MultiIndex columns: (metric_col, stat) e.g. ("psnr_merlin","mean").
stats_df = (
    df.groupby(["lambda", "backend", "arch"])[METRIC_COLS]
    .agg(["mean", "std", "min", "max"])
    .sort_index()  # sort by (lambda, backend, arch)
)

# BPP column: always bpp_bitstream (real rANS) for both GPU and FPGA
BPP_GPU_COL = "bpp_bitstream"
BPP_FPGA_COL = "bpp_bitstream"
missing_bpp = df.query("backend == 'gpu'")["bpp_bitstream"].isna().sum()
if missing_bpp:
    raise ValueError(
        f"{missing_bpp} GPU runs are missing bpp_bitstream. "
        "Run update_wandb_runs.py to compute real bitstream BPP before using this notebook."
    )
print("BPP column for all platforms: bpp_bitstream (real rANS entropy coding)")

# Quick preview: mean PSNR vs MERLIN at each λ for each backend
preview = stats_df["psnr_merlin"]["mean"].unstack(["arch", "backend"])
n_seeds = df["seed"].nunique()
print(f"\nMean PSNR vs MERLIN across {n_seeds} seeds:")
print(preview.to_string(float_format="{:.2f}".format))

## 6 · RD-curve plots — GPU vs FPGA

One curve per `(architecture, backend)` combination.
- **Color** → architecture  |  **Linestyle** → backend (solid=GPU, dashed=FPGA)
- **Band** → min/max range across seeds  |  **Error bars** → ±1 std on BPP

Set `ARCHS_TO_PLOT` in the call cell to control which architectures appear.

In [ ]:
def plot_rd_curve(
    ax: plt.Axes,
    stats: pd.DataFrame,
    backend: str,
    arch: str,
    bpp_col: str,
    quality_col: str,
    label: Optional[str] = None,
    annotate_lambda: bool = False,
    alpha_fill: float = 0.1,
    markersize: float = 3,
) -> None:
    """Plot one RD curve (one backend + architecture) with error band and BPP error bars."""
    bpp_mean = stats[(bpp_col, "mean")].astype(float)
    bpp_std = stats[(bpp_col, "std")].astype(float).fillna(0)
    q_mean = stats[(quality_col, "mean")].astype(float)
    q_min = stats[(quality_col, "min")].astype(float)
    q_max = stats[(quality_col, "max")].astype(float)

    order = bpp_mean.argsort()
    bpp_s = bpp_mean.iloc[order].values
    q_s = q_mean.iloc[order].values
    q_lo = q_min.iloc[order].values
    q_hi = q_max.iloc[order].values
    bpp_std_s = bpp_std.iloc[order].values
    lambdas = stats.index.get_level_values("lambda").to_numpy()[order]

    color = ARCH_COLORS.get(arch, BACKEND_COLORS.get(backend, "#999999"))
    marker = MARKERS[backend]
    ls = LINESTYLES[backend]
    lbl = label or f"{_ARCH_LBL.get(arch, arch)} ({backend.upper()})"

    ax.plot(
        bpp_s,
        q_s,
        marker=marker,
        linestyle=ls,
        color=color,
        label=lbl,
        linewidth=1.5,
        markersize=markersize,
    )
    ax.fill_between(bpp_s, q_lo, q_hi, alpha=alpha_fill, color=color)
    ax.errorbar(
        bpp_s,
        q_s,
        xerr=bpp_std_s,
        fmt="none",
        ecolor=METRIC_COLORS["error_bars"],
        alpha=0.5,
        capsize=2,
    )

    if annotate_lambda:
        for x, yv, lam in zip(bpp_s, q_s, lambdas):
            ax.annotate(
                f"λ={int(lam)}",
                (x, yv),
                textcoords="offset points",
                xytext=(4, 3),
                fontsize=7,
                color=METRIC_COLORS["error_bars"],
                alpha=0.8,
            )


def _parse_quality_col(
    quality_col: Union[str, Tuple[str, str]],
) -> Tuple[str, str]:
    """Return (col_key, y_label) from either a plain string or a (key, label) tuple."""
    if isinstance(quality_col, tuple):
        return quality_col[0], quality_col[1]
    return quality_col, quality_col.replace("_", " ")


def make_rd_figure(
    stats_df: pd.DataFrame,
    archs: Optional[List[str]] = None,
    quality_col: Union[str, Tuple[str, str]] = "psnr_merlin",
    bpp_gpu_col: str = "bpp_bitstream",
    bpp_fpga_col: str = "bpp_bitstream",
    annotate_lambda: bool = False,
    title: Optional[str] = None,
    save_name: Optional[str] = None,
    ax: Optional[plt.Axes] = None,
    alpha_fill: float = 0.1,
    show_legend: bool = True,
    markersize: float = 5,
) -> plt.Figure:
    """RD-curve figure: one curve per (architecture, backend) combination.

    Color encodes architecture (Okabe-Ito palette).
    Linestyle encodes backend (GPU=solid, FPGA=dashed).
    Error band = min/max across seeds.  Error bars = ±1 std on BPP.

    Args:
        archs:       Architectures to plot; ``None`` plots all present in ``stats_df``.
        quality_col: Column key string (e.g. ``"psnr_merlin"``) or ``(key, y_label)`` tuple.
        alpha_fill:  Opacity for the min/max error band (default 0.1).
        show_legend: Whether to draw the automatic combined legend.
        markersize:  Marker size for data points (default 3).
        save_name:   File stem for saving (no extension). Ignored when ``ax`` is provided.
        ax:          Existing Axes for embedding inside a grid. Skips figure creation,
                     ``tight_layout``, and save — the caller manages those.
    """
    col_key, col_label = _parse_quality_col(quality_col)

    _own_fig = ax is None
    if _own_fig:
        fig, ax = plt.subplots(figsize=(9, 6))
    else:
        fig = ax.figure

    # Discover (arch, backend) combinations present in stats_df, optionally filtered
    unique_combos = sorted(
        {
            (arch, backend)
            for _, backend, arch in stats_df.index
            if (archs is None or arch in archs)
        }
    )

    for arch, backend in unique_combos:
        bpp_col = bpp_fpga_col if backend == "fpga" else bpp_gpu_col
        try:
            slice_ = stats_df.xs((backend, arch), level=("backend", "arch"))
        except KeyError:
            continue
        plot_rd_curve(
            ax,
            slice_,
            backend,
            arch,
            bpp_col,
            col_key,
            label=f"{_ARCH_LBL.get(arch, arch)} ({backend.upper()})",
            annotate_lambda=annotate_lambda,
            alpha_fill=alpha_fill,
            markersize=markersize,
        )

    ax.set_xlabel("Bitrate [bpp]", fontsize=8 if not _own_fig else 10)
    ax.set_ylabel(col_label, fontsize=8 if not _own_fig else 10)
    if title or _own_fig:
        ax.set_title(title or col_label, fontsize=8.5 if not _own_fig else 11)
    if show_legend:
        ax.legend(loc="lower right", fontsize=7 if not _own_fig else 9)
    ax.grid(alpha=0.3)

    if _own_fig:
        plt.tight_layout()
        if save_name and SAVE_FIGURES:
            out = PLOTS_DIR / f"{save_name}.pdf"
            fig.savefig(out, bbox_inches="tight")
            print(f"Saved: {out}")
    return fig

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_PLOT = ["ResSHyp", "SHyp", "ResFP", "FP"]  # reduce list for less crowded plots
# ─────────────────────────────────────────────────────────────────────────────
# B7: two legends — color = arch, linestyle = backend (same marker for both)
fig_overlay = make_rd_figure(
    stats_df,
    archs=ARCHS_TO_PLOT,
    quality_col=("psnr_merlin", "PSNR [dB]"),
    bpp_gpu_col=BPP_GPU_COL,
    annotate_lambda=False,
    show_legend=False,  # replaced by two explicit legends below
)
_ax_ov = fig_overlay.axes[0]
_arch_h = [
    Line2D([0], [0], color=ARCH_COLORS[a], lw=2.5, label=_ARCH_LBL.get(a, a))
    for a in ARCHS_TO_PLOT
    if a in ARCH_COLORS
]
_be_h = [
    Line2D([0], [0], color="0.3", marker="o", ls="-", ms=5, label="GPU (FP32)"),
    Line2D([0], [0], color="0.3", marker="o", ls="--", ms=5, label="FPGA (INT8)"),
]
_l1 = _ax_ov.legend(handles=_arch_h, title="architecture", loc="lower right", fontsize=9)
_ax_ov.add_artist(_l1)
_ax_ov.legend(
    handles=_be_h, title="backend", loc="lower right", bbox_to_anchor=(0.80, 0.0), fontsize=9
)
if SAVE_FIGURES:
    fig_overlay.savefig(PLOTS_DIR / "RD-curves_gpu_vs_fpga_psnr.pdf", bbox_inches="tight")
    print(f"Saved: {PLOTS_DIR / 'RD-curves_gpu_vs_fpga_psnr.pdf'}")
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_PLOT = ["ResSHyp", "SHyp", "ResFP", "FP"]  # reduce list for less crowded plots
# ─────────────────────────────────────────────────────────────────────────────
make_rd_figure(
    stats_df,
    archs=ARCHS_TO_PLOT,
    quality_col=("ssim_merlin", "SSIM"),
    bpp_gpu_col=BPP_GPU_COL,
    annotate_lambda=False,
    save_name="RD-curves_gpu_vs_fpga_ssim",
)
plt.show()

In [ ]:
# ── F4 — cross-precision RD, combined 2x4 (PSNR/SSIM rows × arch cols) → paper ──
# B5: shared x/y axis ranges per metric row (auto from data, with 5 % margin)
# B6: clean x-axis label
from matplotlib.lines import Line2D as _Line2D

F4_METRICS = [("psnr_merlin", "PSNR [dB]"), ("ssim_merlin", "SSIM")]
F4_ARCHS = ["ResSHyp", "SHyp", "ResFP", "FP"]
SHARED_XLIM = None  # None = auto from data; or set explicitly e.g. (0.0, 2.0)
SHARED_YLIM = {mk: None for mk, _ in F4_METRICS}  # None = auto per metric row

fig_f4, axes_f4 = plt.subplots(
    len(F4_METRICS),
    len(F4_ARCHS),
    figsize=(3.3 * len(F4_ARCHS), 3.0 * len(F4_METRICS) + 0.4),
    squeeze=False,
)
for _ri, (_mk, _ml) in enumerate(F4_METRICS):
    for _ci, _arch in enumerate(F4_ARCHS):
        _ax = axes_f4[_ri, _ci]
        make_rd_figure(
            stats_df,
            quality_col=(_mk, _ml),
            archs=[_arch],
            bpp_gpu_col=BPP_GPU_COL,
            ax=_ax,
            show_legend=False,
            markersize=4,
        )
        _ax.set_title(
            _ARCH_LBL.get(_arch, _arch) if _ri == 0 else "", fontweight="bold", fontsize=11
        )
        _ax.set_xlabel("Bitrate [bpp]" if _ri == len(F4_METRICS) - 1 else "", fontsize=9)
        _ax.set_ylabel(_ml if _ci == 0 else "", fontsize=9)

# ── B5: Shared axis ranges per metric row ────────────────────────────────────
for _ri, (_mk, _ml) in enumerate(F4_METRICS):
    xlim = SHARED_XLIM
    ylim = SHARED_YLIM.get(_mk)
    if xlim is None or ylim is None:
        _xall, _yall = [], []
        for _ci in range(len(F4_ARCHS)):
            for line in axes_f4[_ri, _ci].get_lines():
                _xall.extend([v for v in line.get_xdata() if np.isfinite(v)])
                _yall.extend([v for v in line.get_ydata() if np.isfinite(v)])
        if xlim is None and _xall:
            _xm, _xM = min(_xall), max(_xall)
            _xp = (_xM - _xm) * 0.05 or 0.05
            xlim = (_xm - _xp, _xM + _xp)
        if ylim is None and _yall:
            _ym, _yM = min(_yall), max(_yall)
            _yp = (_yM - _ym) * 0.05 or 0.5
            ylim = (_ym - _yp, _yM + _yp)
    for _ci in range(len(F4_ARCHS)):
        if xlim:
            axes_f4[_ri, _ci].set_xlim(*xlim)
        if ylim:
            axes_f4[_ri, _ci].set_ylim(*ylim)

_h = [
    _Line2D([0], [0], color="0.3", marker="o", ls="-", label="GPU (FP32)"),
    _Line2D([0], [0], color="0.3", marker="o", ls="--", label="FPGA (INT8)"),
]
fig_f4.legend(handles=_h, loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))
fig_f4.tight_layout(rect=(0, 0.04, 1, 1))
if SAVE_FIGURES:
    fig_f4.savefig(PLOTS_DIR / "RD-curves_crossprecision_2x4.pdf", bbox_inches="tight")
export_manuscript(fig_f4, "fig_crossprecision_RD")
plt.show()

## §6.3 · Numerical GPU vs FPGA comparison at fixed λ

Per-architecture comparison of GPU (FP32) vs FPGA (INT8) at a chosen λ.
Reports mean±std for each backend and mean±SE of the difference (GPU − FPGA), where:

    SE(Δ) = √(var_GPU / n_GPU + var_FPGA / n_FPGA)   [Welch — independent groups]

The two groups (6 GPU seeds, 6 FPGA seeds) are **independent**: same training run,
different inference precision. A positive Δ means GPU is better than FPGA.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
COMPARE_LAMBDA = 1000.0  # change to any λ present in the dataset
ARCHS_TO_COMPARE = ["ResSHyp", "SHyp", "ResFP", "FP"]
# ─────────────────────────────────────────────────────────────────────────────
# df uses "lambda" column and flat metric names (no prefix) — _plotkit defaults apply.
for _arch in ARCHS_TO_COMPARE:
    compare_at_lambda(
        df[(df["backend"] == "gpu") & (df["arch"] == _arch)],
        df[(df["backend"] == "fpga") & (df["arch"] == _arch)],
        f"{_ARCH_LBL.get(_arch, _arch)} GPU",
        f"{_ARCH_LBL.get(_arch, _arch)} FPGA",
        lmbda=COMPARE_LAMBDA,
    )

In [ ]:
# ── EPD (Edge Preservation Degree) GPU vs FPGA at λ — tracks the manuscript §V-C number ──
# compare_at_lambda above reports PSNR/SSIM/bpp; EPD is not plotted, so we print its Δ here
# to back the loss-of-fine-structure interpretation (higher EPD = better edge preservation).
print(f"EPD vs MERLIN at λ={int(COMPARE_LAMBDA)}  (Δ = GPU − FPGA):")
_epd_deltas = []
for _arch in ARCHS_TO_COMPARE:
    _g = df[(df["backend"] == "gpu") & (df["arch"] == _arch) & (df["lambda"] == COMPARE_LAMBDA)][
        "epd_merlin"
    ].dropna()
    _f = df[(df["backend"] == "fpga") & (df["arch"] == _arch) & (df["lambda"] == COMPARE_LAMBDA)][
        "epd_merlin"
    ].dropna()
    _d = _g.mean() - _f.mean()
    _se = np.sqrt(_g.var(ddof=1) / len(_g) + _f.var(ddof=1) / len(_f))
    _epd_deltas.append(_d)
    print(
        f"  {_ARCH_LBL.get(_arch, _arch):<6}  GPU {_g.mean():.3f}±{_g.std(ddof=1):.3f}   "
        f"FPGA {_f.mean():.3f}±{_f.std(ddof=1):.3f}   Δ {_d:+.3f}±{_se:.3f}"
    )
print(f"  → Δ range across architectures: {min(_epd_deltas):.2f} to {max(_epd_deltas):.2f}")

## §6.5 · All-metrics RD grid

Full quality overview across all tracked metrics for each architecture.
Edit `ARCHS_TO_PLOT` and `_METRIC_LABELS` below to control the output.

In [ ]:
# Human-readable Y-axis labels.  Comment out entries to hide them from the grid.
_METRIC_LABELS: Dict[str, str] = {
    "psnr_merlin": "PSNR [dB]",
    "ssim_merlin": "SSIM",
    "epd_merlin": "EPD",
    "ratio_mean": "Mean(noisy_I / recon_I)",
    "ratio_enl": "ENL of Ratio image",
    "enl_recon": "ENL of Reconstruction",
    # "psnr_adam_noc": "PSNR vs ADAM-NOC [dB]",
    # "epd_adam_noc":  "EPD vs ADAM-NOC",
}
N_COLS_GRID = 3  # 2 or 3 — figure grows taller rather than wider

# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_PLOT = ["ResSHyp", "SHyp", "ResFP", "FP"]
# ─────────────────────────────────────────────────────────────────────────────
n_metrics = len(_METRIC_LABELS)
n_rows_grid = (n_metrics + N_COLS_GRID - 1) // N_COLS_GRID

for _arch in ARCHS_TO_PLOT:
    _arch_df = df.query(f"arch == '{_arch}'")
    if _arch_df.empty:
        print(f"{y}No data for {_arch} — skipping.{e}")
        continue

    fig_all, axes_all = plt.subplots(
        n_rows_grid,
        N_COLS_GRID,
        figsize=(N_COLS_GRID * 5.5, n_rows_grid * 4.2),
        squeeze=False,
    )

    for idx, (metric_col, metric_label) in enumerate(_METRIC_LABELS.items()):
        _r, _c = divmod(idx, N_COLS_GRID)
        make_rd_figure(
            stats_df,  # use aggregated stats (MultiIndex), not raw _arch_df
            quality_col=(metric_col, metric_label),
            bpp_gpu_col=BPP_GPU_COL,
            archs=[_arch],
            annotate_lambda=False,
            ax=axes_all[_r, _c],
        )

    for idx in range(n_metrics, n_rows_grid * N_COLS_GRID):
        _r, _c = divmod(idx, N_COLS_GRID)
        axes_all[_r, _c].axis("off")

    fig_all.suptitle(
        f"GPU vs FPGA — All Quality Metrics  —  {_ARCH_LBL.get(_arch, _arch)}",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()

    if SAVE_FIGURES:
        out_all = PLOTS_DIR / f"rd_all_metrics_{_arch}.pdf"
        fig_all.savefig(out_all, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_all}")
    plt.show()

## Legacy / Exploratory

### §7 · FPGA degradation Δ

Per-seed FPGA − GPU quality delta per λ. Negative = FPGA is worse than GPU.

In [ ]:
def compute_delta(df: pd.DataFrame, quality_col: str = "psnr_merlin") -> pd.DataFrame:
    """Compute per-seed FPGA − GPU delta for a quality metric, then aggregate.

    Returns a DataFrame indexed by lambda with columns mean/std/min/max.
    """
    pivot = df.pivot_table(index=["lambda", "seed"], columns="backend", values=quality_col)
    pivot = pivot.dropna(subset=["gpu", "fpga"])
    pivot["delta"] = pivot["fpga"] - pivot["gpu"]
    return pivot["delta"].groupby("lambda").agg(["mean", "std", "min", "max"]).sort_index()


def plot_delta(
    deltas: List[Tuple[pd.DataFrame, str]],
    bpp_means: pd.Series,
    arch: str,
    yaxis_label: Optional[str] = None,
    save_name: Optional[str] = None,
) -> plt.Figure:
    """Bar chart of mean FPGA degradation per λ.

    Args:
        deltas:     List of ``(delta_df, label)`` tuples from ``compute_delta()``.
                    Single element → per-bar sign colouring.
                    Multiple elements → grouped bars, one colour per metric.
        bpp_means:  Mean GPU BPP per λ (for X-tick labels).
        save_name:  File stem for saving (no extension).
    """
    n_metrics = len(deltas)
    all_lambdas = sorted(set().union(*[set(d.index) for d, _ in deltas]))
    lambdas = np.array(all_lambdas, dtype=float)
    x = np.arange(len(lambdas))
    group_w = 0.70
    width = group_w / n_metrics
    offsets = np.linspace(-group_w / 2 + width / 2, group_w / 2 - width / 2, n_metrics)

    _metric_palette = [METRIC_COLORS["psnr"], METRIC_COLORS["ssim"], METRIC_COLORS["epd"]]

    fig, ax = plt.subplots(figsize=(max(10, 1.5 * len(lambdas)), 4))

    for i, (delta, label) in enumerate(deltas):
        means = np.array(
            [delta.loc[l, "mean"] if l in delta.index else float("nan") for l in lambdas]
        )
        mins = np.array(
            [delta.loc[l, "min"] if l in delta.index else float("nan") for l in lambdas]
        )
        maxs = np.array(
            [delta.loc[l, "max"] if l in delta.index else float("nan") for l in lambdas]
        )
        pos = x + offsets[i]

        bar_color = (
            BACKEND_COLORS["fpga"] if n_metrics == 1 else _metric_palette[i % len(_metric_palette)]
        )
        ax.bar(pos, means, width, color=bar_color, alpha=0.75, label=label)
        ax.errorbar(
            pos,
            means,
            yerr=[np.nan_to_num(means - mins), np.nan_to_num(maxs - means)],
            fmt="none",
            color=METRIC_COLORS["error_bars"],
            capsize=3,
            alpha=0.7,
        )

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"λ={int(l)}\n({bpp_means.get(l, float('nan')):.3f} bpp)" for l in lambdas],
        fontsize=9,
    )
    ax.set_ylabel(yaxis_label or "Difference FPGA − GPU")
    ax.set_title(
        f"FPGA quantization degradation  —  {arch}\n(negative = FPGA worse than GPU)",
        fontsize=10,
    )
    ax.grid(axis="y", alpha=0.3)
    if n_metrics > 1:
        ax.legend(fontsize=8)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        out = PLOTS_DIR / f"{save_name}.pdf"
        fig.savefig(out, bbox_inches="tight")
        print(f"Saved: {out}")
    return fig


# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_FOR_DELTA = ["ResSHyp", "SHyp", "ResFP", "FP"]
DELTA_METRICS = [
    ("psnr_merlin", "PSNR [dB]"),
    ("ssim_merlin", "SSIM"),
    ("epd_merlin", "EPD"),
]
# ─────────────────────────────────────────────────────────────────────────────
for _arch in ARCHS_FOR_DELTA:
    _arch_df = df.query(f"arch == '{_arch}'")
    if _arch_df.empty:
        print(f"{y}No data for {_arch} — skipping.{e}")
        continue

    _deltas = [(compute_delta(_arch_df, col), label) for col, label in DELTA_METRICS]
    _bpp_by_lam = _arch_df.query("backend == 'gpu'").groupby("lambda")[BPP_GPU_COL].mean()

    print(f"\n{_arch} — average FPGA degradation across all λ:")
    for (col, label), (delta, _) in zip(DELTA_METRICS, _deltas):
        print(f"  {label}: {delta['mean'].mean():.3f}")

    plot_delta(
        _deltas,
        _bpp_by_lam,
        arch=_arch,
        yaxis_label=f"Quality difference FPGA − GPU  —  {_arch}",
        save_name=f"FPGA_degradation_{_arch}",
    )
    plt.show()

## 8 · BPP comparison — likelihood vs rANS bitstream

The GPU trains with a soft entropy estimate (likelihood BPP from the entropy bottleneck).
The FPGA uses real rANS coding. This plot shows how they compare at each λ.

- **If `update_wandb_runs.py` has run and added `bpp_bitstream` to W&B**: both backends
  show hard BPP and the comparison is apples-to-apples.
- **Currently**: GPU shows likelihood BPP (soft, usually slightly lower than real coding)
  and FPGA shows rANS BPP. The gap is the coding overhead of the real entropy coder.

> To add `bpp_bitstream` to the GPU runs: complete `update_wandb_runs.py` and re-run cell 2.
> `BPP_GPU_COL` will automatically switch to `"bpp_bitstream"`.


In [ ]:
def plot_bpp_comparison(
    df: pd.DataFrame,
    arch: str,
    save_name: Optional[str] = None,
) -> plt.Figure:
    """Grouped bar chart: GPU likelihood BPP vs GPU bitstream BPP vs FPGA rANS BPP per λ."""
    bpp_stats = df.groupby(["lambda", "backend"]).agg(
        bpp_gpu_lik_mean=("bpp_likelihood", "mean"),
        bpp_gpu_lik_std=("bpp_likelihood", "std"),
        bpp_bitstream_mean=("bpp_bitstream", "mean"),
        bpp_bitstream_std=("bpp_bitstream", "std"),
    )

    lambdas = sorted(df["lambda"].unique())
    x = np.arange(len(lambdas))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 5))

    gpu_lik_means, gpu_lik_stds = [], []
    gpu_bit_means, gpu_bit_stds = [], []
    fpga_means, fpga_stds = [], []

    for lam in lambdas:
        try:
            g_row = bpp_stats.loc[(lam, "gpu")]
            gpu_lik_means.append(g_row.get("bpp_gpu_lik_mean", float("nan")))
            gpu_lik_stds.append(g_row.get("bpp_gpu_lik_std", 0) or 0)
            gpu_bit_means.append(g_row.get("bpp_bitstream_mean", float("nan")))
            gpu_bit_stds.append(g_row.get("bpp_bitstream_std", 0) or 0)
        except KeyError:
            gpu_lik_means.append(float("nan"))
            gpu_lik_stds.append(0)
            gpu_bit_means.append(float("nan"))
            gpu_bit_stds.append(0)
        try:
            f_row = bpp_stats.loc[(lam, "fpga")]
            fpga_means.append(f_row.get("bpp_bitstream_mean", float("nan")))
            fpga_stds.append(f_row.get("bpp_bitstream_std", 0) or 0)
        except KeyError:
            fpga_means.append(float("nan"))
            fpga_stds.append(0)

    ax.bar(
        x - width,
        gpu_lik_means,
        width,
        label="GPU likelihood BPP",
        color=PALETTE["platforms"]["gpu_idle"],
        yerr=gpu_lik_stds,
        capsize=2,
    )
    ax.bar(
        x,
        gpu_bit_means,
        width,
        label="GPU bitstream BPP",
        color=PALETTE["platforms"]["gpu_dynamic"],
        yerr=gpu_bit_stds,
        capsize=2,
    )
    ax.bar(
        x + width,
        fpga_means,
        width,
        label="FPGA rANS BPP",
        color=PALETTE["platforms"]["fpga_dynamic"],
        yerr=fpga_stds,
        capsize=2,
    )

    ax.set_xticks(x)
    ax.set_xticklabels([f"λ={int(l)}" for l in lambdas], fontsize=8)
    ax.set_ylabel("Bitrate [bpp]")
    ax.set_title(f"BPP comparison: GPU likelihood / GPU bitstream / FPGA rANS  —  {arch}")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        out = PLOTS_DIR / f"{save_name}.pdf"
        fig.savefig(out, bbox_inches="tight")
        print(f"Saved: {out}")
    return fig


# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_FOR_BPP = ["ResSHyp", "SHyp", "ResFP", "FP"]
# ─────────────────────────────────────────────────────────────────────────────
for _arch in ARCHS_FOR_BPP:
    _arch_df = df.query(f"arch == '{_arch}'")
    if _arch_df.empty:
        print(f"{y}No data for {_arch} — skipping.{e}")
        continue
    plot_bpp_comparison(_arch_df, arch=_arch, save_name=f"BPP_comparison_{_arch}")
    plt.show()

## §9 · Hamburg Tile Visualization → `reconstruction_visualization.ipynb`

Hamburg tile reconstruction grids have been moved to the dedicated notebook
`notebooks/reconstruction_visualization.ipynb`, which loads data independently
via `_plotkit.load_quality_runs` and `load_fpga_quality`.